## **Library**

In [ ]:
!pip install google-play-scraper nltk Sastrawi pyLDAvis -q


In [ ]:
import pandas as pd
import numpy as np
import re
import time
import nltk
import matplotlib.pyplot as plt
import pyLDAvis
import pyLDAvis.lda_model

from google_play_scraper import reviews, Sort
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from wordcloud import WordCloud

nltk.download("stopwords")

import warnings
warnings.filterwarnings("ignore")

## **Data Scraping**

In [ ]:
APPS = {
    "Mobile JKN": "app.bpjs.mobile",
    "SATUSEHAT": "com.telkom.tracencare"
}

TARGET_PER_APP = 2000
BATCH_SIZE = 200

all_data = []

for app_name, app_id in APPS.items():
    print(f"scraping: {app_name}")
    print(f"App ID: {app_id}")
    
    all_reviews = []
    continuation_token = None
    
    while len(all_reviews) < TARGET_PER_APP:
        try:
            result, continuation_token = reviews(
                app_id,
                lang="id",
                country="id",
                sort=Sort.NEWEST,
                count=BATCH_SIZE,
                continuation_token=continuation_token
            )
            
            if len(result) == 0:
                print("Tidak ada review tambahan.")
                break
            
            all_reviews.extend(result)
            print(f"{app_name} - Total review terkumpul: {len(all_reviews)}")
            
            if continuation_token is None:
                break
            
            time.sleep(1)
        
        except Exception as e:
            print(f"Error saat scraping {app_name}: {e}")
            break
    
    df_app = pd.DataFrame(all_reviews)
    
    if len(df_app) == 0:
        print(f"Tidak ada data untuk {app_name}")
        continue
    
    df_app = df_app[[
        "reviewId",
        "userName",
        "content",
        "score",
        "thumbsUpCount",
        "reviewCreatedVersion",
        "at",
        "replyContent",
        "repliedAt",
        "appVersion"
    ]]
    
    df_app = df_app.rename(columns={
        "content": "review",
        "score": "rating",
        "at": "tanggal"
    })
    
    df_app["source_app"] = app_name
    df_app["app_id"] = app_id
    
    all_data.append(df_app)

# Gabungkan semua app
df = pd.concat(all_data, ignore_index=True)

# Bersihkan data dasar
df = df.dropna(subset=["review"])
df["review"] = df["review"].astype(str)
df = df.drop_duplicates(subset=["source_app", "review"])

df["tanggal"] = pd.to_datetime(df["tanggal"], errors="coerce")

# Simpan data mentah gabungan
df.to_csv("data/raw/review_mobile_jkn_satusehat_raw.csv", index=False, encoding="utf-8-sig")

print("\nSelesai scraping!")
print("Jumlah data final:", len(df))
print("\nJumlah review per aplikasi:")
display(df["source_app"].value_counts().reset_index().rename(columns={
    "index": "Aplikasi",
    "source_app": "Jumlah Review"
}))

display(df.head())

## **Text Preprocessing**

In [ ]:
stop_words = set(stopwords.words("indonesian"))

custom_stopwords = {
    # kata umum
    "aplikasi", "app", "apk", "nya", "yg", "yang", "ga", "gak", "nggak",
    "tdk", "tidak", "aja", "sih", "dong", "min", "admin", "banget",
    "bgt", "tolong", "mohon", "saya", "aku", "kami", "kita", "nih",
    "deh", "kok", "lah", "pun", "kan", "ya", "udah", "sudah", "orang",
    "bikin", "buat", "pake", "pakai", "kali", "gitu", "coba", "kasih",
    "mantap", "men",
    
    # nama aplikasi / domain umum
    "bpjs", "jkn", "mobile", "satu", "sehat", "satusehat", "pedulilindungi",
    
    # kata terlalu umum untuk topic modeling
    "bagus", "baik", "buruk", "jelek", "terima", "kasih", "terimakasih"
}

stop_words.update(custom_stopwords)

factory = StemmerFactory()
stemmer = factory.create_stemmer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    
    stemmed = [stemmer.stem(word) for word in tokens]
    
    return " ".join(stemmed)

df["clean_review"] = df["review"].apply(clean_text)
df = df[df["clean_review"].str.strip() != ""]

df["word_count_raw"] = df["review"].apply(lambda x: len(str(x).split()))
df["word_count_clean"] = df["clean_review"].apply(lambda x: len(str(x).split()))

df.to_csv("data/processed/review_mobile_jkn_satusehat_clean.csv", index=False, encoding="utf-8-sig")

print("\nJumlah data setelah preprocessing:", len(df))
print("\nJumlah data bersih per aplikasi:")
display(df["source_app"].value_counts().reset_index().rename(columns={
    "index": "Aplikasi",
    "source_app": "Jumlah Review"
}))

display(df[["source_app", "review", "clean_review", "rating", "tanggal"]].head(10))

## **EDA**

In [ ]:
# Color pallete
APP_COLORS    = ["#4C9A8A", "#E07B54"]  
SINGLE_COLOR  = "#4C9A8A"               
HIST_COLOR    = "#7FB5AC"                
TOPIC_COLOR   = "#5B86A8"               
WC_COLORMAP   = "GnBu"                   

# Distribusi jumlah review per aplikasi
plt.figure(figsize=(7, 5))
counts = df["source_app"].value_counts()
bars = plt.bar(counts.index, counts.values, color=APP_COLORS[:len(counts)])
plt.title("Jumlah Review per Aplikasi")
plt.xlabel("Aplikasi")
plt.ylabel("Jumlah Review")
plt.xticks(rotation=0)
plt.show()

# Distribusi rating per aplikasi
rating_table = pd.crosstab(df["rating"], df["source_app"])

rating_table.plot(kind="bar", figsize=(9, 5), color=APP_COLORS)
plt.title("Perbandingan Distribusi Rating Mobile JKN vs SATUSEHAT")
plt.xlabel("Rating")
plt.ylabel("Jumlah Review")
plt.xticks(rotation=0)
plt.show()

display(rating_table)

# Distribusi panjang review setelah preprocessing
for app_name in df["source_app"].unique():
    subset = df[df["source_app"] == app_name]
    
    plt.figure(figsize=(8, 5))
    plt.hist(subset["word_count_clean"], bins=30, color=HIST_COLOR, edgecolor="white")
    plt.title(f"Distribusi Panjang Review Setelah Preprocessing - {app_name}")
    plt.xlabel("Jumlah Kata")
    plt.ylabel("Frekuensi")
    plt.show()

# Wordcloud per aplikasi
for app_name in df["source_app"].unique():
    subset = df[df["source_app"] == app_name]
    all_words = " ".join(subset["clean_review"])
    
    wordcloud = WordCloud(
        width=1000,
        height=500,
        background_color="white",
        colormap=WC_COLORMAP,
        collocations=False
    ).generate(all_words)
    
    plt.figure(figsize=(12, 6))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Word Cloud Review - {app_name}")
    plt.show()

# Top words per aplikasi
def get_top_words(text_series, max_features=20):
    vectorizer_top = CountVectorizer(max_features=max_features)
    matrix = vectorizer_top.fit_transform(text_series)
    
    top_words = pd.DataFrame({
        "kata": vectorizer_top.get_feature_names_out(),
        "frekuensi": matrix.toarray().sum(axis=0)
    }).sort_values(by="frekuensi", ascending=False)
    
    return top_words

top_words_all = []

for app_name in df["source_app"].unique():
    subset = df[df["source_app"] == app_name]
    top_words = get_top_words(subset["clean_review"], 20)
    top_words["source_app"] = app_name
    top_words_all.append(top_words)
    
    plt.figure(figsize=(10, 6))
    plt.barh(top_words["kata"], top_words["frekuensi"], color=SINGLE_COLOR)
    plt.gca().invert_yaxis()
    plt.title(f"20 Kata Paling Sering Muncul - {app_name}")
    plt.xlabel("Frekuensi")
    plt.ylabel("Kata")
    plt.show()
    
    display(top_words)

top_words_all = pd.concat(top_words_all, ignore_index=True)
top_words_all.to_csv("data/processed/top_words_mobile_jkn_satusehat.csv", index=False, encoding="utf-8-sig")

# Bigram per aplikasi
def get_top_ngrams(text_series, ngram_range=(2, 2), max_features=20):
    vectorizer_ngram = CountVectorizer(
        ngram_range=ngram_range,
        max_features=max_features
    )
    matrix = vectorizer_ngram.fit_transform(text_series)
    
    ngrams = pd.DataFrame({
        "ngram": vectorizer_ngram.get_feature_names_out(),
        "frekuensi": matrix.toarray().sum(axis=0)
    }).sort_values(by="frekuensi", ascending=False)
    
    return ngrams

bigram_all = []

for app_name in df["source_app"].unique():
    subset = df[df["source_app"] == app_name]
    bigrams = get_top_ngrams(subset["clean_review"], (2, 2), 20)
    bigrams["source_app"] = app_name
    bigram_all.append(bigrams)
    
    plt.figure(figsize=(10, 6))
    plt.barh(bigrams["ngram"], bigrams["frekuensi"], color=SINGLE_COLOR)
    plt.gca().invert_yaxis()
    plt.title(f"20 Bigram Paling Sering Muncul - {app_name}")
    plt.xlabel("Frekuensi")
    plt.ylabel("Bigram")
    plt.show()
    
    display(bigrams)

bigram_all = pd.concat(bigram_all, ignore_index=True)
bigram_all.to_csv("data/processed/bigram_mobile_jkn_satusehat.csv", index=False, encoding="utf-8-sig")

## **Topic Modelling**

In [ ]:
NUM_TOPICS = 4

all_topic_tables = []
all_topic_counts = []
df_result_list = []

def run_lda_for_app(data, app_name, num_topics=4):
    print(f"TOPIC MODELING LDA - {app_name}")
    
    vectorizer = CountVectorizer(
        max_df=0.85,
        min_df=10,
        max_features=3000,
        ngram_range=(1, 2)
    )
    
    dtm = vectorizer.fit_transform(data["clean_review"])
    
    lda_model = LatentDirichletAllocation(
        n_components=num_topics,
        random_state=42,
        learning_method="batch",
        max_iter=30
    )
    
    lda_model.fit(dtm)
    
    feature_names = vectorizer.get_feature_names_out()
    
    topic_list = []
    
    for topic_idx, topic in enumerate(lda_model.components_):
        top_indices = np.argsort(topic)[-12:][::-1]
        top_words = [feature_names[i] for i in top_indices]
        
        topic_list.append({
            "source_app": app_name,
            "topic_number": topic_idx + 1,
            "top_words": ", ".join(top_words)
        })
        
        print(f"\nTopic {topic_idx + 1}:")
        print(", ".join(top_words))
    
    topic_df = pd.DataFrame(topic_list)
    
    topic_distribution = lda_model.transform(dtm)
    
    data = data.copy()
    data["dominant_topic"] = topic_distribution.argmax(axis=1) + 1
    data["topic_probability"] = topic_distribution.max(axis=1)
    
    topic_counts = data["dominant_topic"].value_counts().sort_index().reset_index()
    topic_counts.columns = ["topic_number", "jumlah_review"]
    topic_counts["source_app"] = app_name
    topic_counts["persentase"] = topic_counts["jumlah_review"] / topic_counts["jumlah_review"].sum() * 100
    
    plt.figure(figsize=(8, 5))
    plt.bar(topic_counts["topic_number"].astype(str), topic_counts["jumlah_review"], color=TOPIC_COLOR)
    plt.title(f"Distribusi Review Berdasarkan Dominant Topic - {app_name}")
    plt.xlabel("Topic")
    plt.ylabel("Jumlah Review")
    plt.show()
    
    display(topic_df)
    display(topic_counts)
    
    # Contoh review per topik
    for topic_num in range(1, num_topics + 1):
        print(f"CONTOH REVIEW {app_name} - TOPIC {topic_num}")
        
        sample_reviews = data[data["dominant_topic"] == topic_num] \
            .sort_values(by="topic_probability", ascending=False) \
            [["source_app", "review", "clean_review", "topic_probability"]] \
            .head(5)
        
        display(sample_reviews)
    
    return data, topic_df, topic_counts

for app_name in df["source_app"].unique():
    subset = df[df["source_app"] == app_name].copy()
    
    result_data, topic_table, topic_count = run_lda_for_app(
        subset,
        app_name,
        NUM_TOPICS
    )
    
    df_result_list.append(result_data)
    all_topic_tables.append(topic_table)
    all_topic_counts.append(topic_count)

df_result = pd.concat(df_result_list, ignore_index=True)
topic_table_final = pd.concat(all_topic_tables, ignore_index=True)
topic_count_final = pd.concat(all_topic_counts, ignore_index=True)


# distribusi topik

comparison_topic = topic_count_final.pivot(
    index="topic_number",
    columns="source_app",
    values="persentase"
).fillna(0)

comparison_topic.plot(kind="bar", figsize=(9, 5), color=APP_COLORS)
plt.title("Perbandingan Persentase Dominant Topic per Aplikasi")
plt.xlabel("Topic")
plt.ylabel("Persentase Review (%)")
plt.xticks(rotation=0)
plt.show()

display(comparison_topic)